# Full ACC Pipeline — Local Version (Overpass-first, ElevenLabs voice)

Same pipeline as before, but **road type, surface damage, posted speed limit, stop signs, crosswalks, and speed bumps are looked up from OpenStreetMap (via the Overpass API) using the vehicle's GPS coordinates first** — the corresponding camera model only runs when OSM has no tag for that fact at this exact location. Four things always stay camera-based no matter what, because there's no map equivalent for them: red light *state* (OSM only knows a signal exists there, never its current color), live pedestrian presence, safety distance to the car ahead, and weather. This makes the pipeline lighter (fewer model calls, fewer Roboflow network round-trips per frame) and smarter (OSM's road classification is more reliable than a MobileNetV2 guess from a single frame, when the tag exists).

## 1. Install dependencies

In [1]:
!pip install audioop-lts pydub mutagen tensorflow roboflow ultralytics openai-whisper elevenlabs google-genai sounddevice scipy pillow matplotlib



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports

In [2]:
import os
import re
import random
import requests
import numpy as np
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt

import whisper
from ultralytics import YOLO
from roboflow import Roboflow

from google.genai import types
from elevenlabs.client import ElevenLabs
from elevenlabs.play import save as save_audio

import sounddevice as sd
from scipy.io.wavfile import write as write_wav

from IPython.display import Audio, display
import tkinter as tk
from tkinter import filedialog
from dotenv import load_dotenv



## 3. Config

In [4]:
IMG_SIZE = (224, 224)
CLASS_NAMES = ["autoroute", "urbaine", "rurale"]  # must match the order used during training

LOCAL_MODEL_PATH = "road_classifier.keras"

load_dotenv()
ROBOFLOW_API_KEY = os.getenv("ROBOFLOW_API_KEY")
ELEVENLABS_API_KEY = os.getenv("ELEVENLABS_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")


# --- Roboflow off-road model ---
OFFROAD_WORKSPACE = "sri-lab"
OFFROAD_PROJECT = "road-surface-classification-lgxl1"
OFFROAD_VERSION = 1  # UNVERIFIED — confirm on the project's Universe page
UNPAVED_CLASS_NAMES = {"unpaved"}  # UNVERIFIED — confirm the real class string via a test prediction
OFFROAD_CONFIDENCE_THRESHOLD = 0.5

# --- Roboflow damage model ---
DAMAGE_WORKSPACE = "roaddamage-msfnj"
DAMAGE_PROJECT = "road-damage-ww8ex"
DAMAGE_VERSION = 1  # UNVERIFIED — confirm on the project's Universe page
DAMAGE_CONFIDENCE_THRESHOLD = 0.5

# --- Traffic sign model (GitHub) ---
SIGN_REPO_URL = "https://github.com/bhaskrr/traffic-sign-detection-using-yolov11.git"
SIGN_REPO_LOCAL_DIR = "traffic_sign_repo"
SIGN_WEIGHTS_FILENAME = "traffic_sign_detector.pt"  # confirmed real filename in the repo
SIGN_CONFIDENCE_THRESHOLD = 0.5
IGNORED_SIGN_CLASSES = {"all"}  # known bogus class from the source dataset export

# --- ElevenLabs voice ---
# Default voice_id below ("George") works fine for French via the multilingual model +
# language_code — but browse elevenlabs.io's Voice Library for one you actually like,
# or run elevenlabs_client.voices.search() to list what's available on your account.
ELEVENLABS_VOICE_ID = "JBFqnCBsd6RMkjVDRZzb"
ELEVENLABS_MODEL_ID = "eleven_multilingual_v2"


# --- Speed bump detector (your own trained model — download .pt from Kaggle Output) ---
SPEED_BUMP_MODEL_PATH = "speed_bump_detector.pt"
SPEED_BUMP_CONFIDENCE_THRESHOLD = 0.5

# --- Pedestrian + crosswalk detector (your own trained model — download .pt from Kaggle Output) ---
PED_CROSSWALK_MODEL_PATH = "ped_crosswalk_detector.pt"
PEDESTRIAN_CONFIDENCE_THRESHOLD = 0.5
CROSSWALK_CONFIDENCE_THRESHOLD = 0.5
# Same bbox-size/position heuristic as the safety-distance check below — calibrate against your
# own sample frames, don't trust these placeholders as-is.
PEDESTRIAN_CLOSE_RELATIVE_HEIGHT_THRESHOLD = 0.30  # UNVERIFIED placeholder
PEDESTRIAN_CLOSE_RELATIVE_BOTTOM_THRESHOLD = 0.80  # UNVERIFIED placeholder

# --- Weather classifier (your own trained model, .tflite — download from Kaggle Output) ---
WEATHER_MODEL_PATH = "weather_classifier.tflite"
WEATHER_CLASSES = ["clear", "adverse"]  # must match TARGET_CLASSES order from training
WEATHER_CONFIDENCE_THRESHOLD = 0.7  # only trust an "adverse" call above this; otherwise default to clear

# --- Vehicle detector for safety distance (stock COCO YOLOv8n — auto-downloads, no upload needed) ---
VEHICLE_CLASSES = ["car", "truck", "bus"]
VEHICLE_CONFIDENCE_THRESHOLD = 0.4
CLOSE_RELATIVE_HEIGHT_THRESHOLD = 0.35  # UNVERIFIED placeholder — calibrate against your own frames
CLOSE_RELATIVE_BOTTOM_THRESHOLD = 0.85  # UNVERIFIED placeholder — calibrate against your own frames


# --- Overpass / OpenStreetMap: primary source of truth for road type, surface damage, posted
# speed limit, stop signs, crosswalks, and speed bumps. The camera model for each of these only
# runs when OSM has no matching tag at this location -- see get_overpass_context() below.
OVERPASS_URL = "https://overpass-api.de/api/interpreter"
OVERPASS_SEARCH_RADIUS_M = 40          # how far to look for the road segment itself
OVERPASS_FEATURE_RADIUS_M = 25         # how far to look for stop signs / crossings / speed bumps
OVERPASS_CACHE_PRECISION = 4           # ~11m coordinate rounding -- avoids re-querying every frame
OVERPASS_TIMEOUT_SECONDS = 10

# highway tag -> our 3 road classes. Heuristic -- OSM's tagging scheme doesn't map 1:1 onto
# autoroute/urbaine/rurale, this is a reasonable approximation, not ground truth.
AUTOROUTE_HIGHWAY_TAGS = {"motorway", "motorway_link", "trunk", "trunk_link"}
URBAINE_HIGHWAY_TAGS = {
    "residential", "living_street", "service", "pedestrian",
    "primary", "primary_link", "secondary", "secondary_link", "tertiary", "tertiary_link",
}
RURALE_HIGHWAY_TAGS = {"unclassified", "track", "path"}

UNPAVED_SURFACE_TAGS = {"unpaved", "dirt", "gravel", "sand", "ground", "grass", "mud", "earth", "compacted"}
BAD_SMOOTHNESS_TAGS = {"bad", "very_bad", "horrible", "impassable"}
GOOD_SMOOTHNESS_TAGS = {"excellent", "good", "intermediate"}


# --- Google Street View Static API: stand-in for a live camera frame, fetched by coordinates.
# Needs a Google Cloud project with billing enabled (Maps Platform gives a monthly free credit).
# This is a STATIC, possibly outdated photo (whenever Google's car last drove that street), not
# a live feed -- fine for testing the pipeline end-to-end, not a substitute for a real camera.
GOOGLE_MAPS_API_KEY = "YOUR_GOOGLE_MAPS_API_KEY_HERE"
STREET_VIEW_URL = "https://maps.googleapis.com/maps/api/streetview"
STREET_VIEW_METADATA_URL = "https://maps.googleapis.com/maps/api/streetview/metadata"


## 4. Load the trained road-type classifier

In [5]:
if not os.path.exists(LOCAL_MODEL_PATH):
    raise FileNotFoundError(
        f"{LOCAL_MODEL_PATH} not found. Download it from your Kaggle notebook's Output tab "
        f"and place it next to this notebook."
    )

road_classifier = tf.keras.models.load_model(LOCAL_MODEL_PATH)
print("Road-type classifier loaded.")


Road-type classifier loaded.


## 5. Load the off-road (Roboflow) model

In [6]:
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

offroad_project = rf.workspace(OFFROAD_WORKSPACE).project(OFFROAD_PROJECT)
offroad_model = offroad_project.version(OFFROAD_VERSION).model
print("Off-road model loaded.")


loading Roboflow workspace...
loading Roboflow project...
Off-road model loaded.


In [7]:
def detect_offroad(image_path):
    result = offroad_model.predict(image_path).json()
    predictions = result.get("predictions", [])
    if not predictions:
        return False, 0.0
    top = predictions[0] if isinstance(predictions, list) else predictions
    predicted_class = top.get("class") or top.get("top", "")
    confidence = top.get("confidence", 0.0)
    is_offroad = predicted_class in UNPAVED_CLASS_NAMES and confidence >= OFFROAD_CONFIDENCE_THRESHOLD
    return is_offroad, confidence

## 6. Load the damage (Roboflow) model

In [8]:
damage_project = rf.workspace(DAMAGE_WORKSPACE).project(DAMAGE_PROJECT)
damage_model_rf = damage_project.version(DAMAGE_VERSION).model
print("Damage model loaded.")


loading Roboflow workspace...
loading Roboflow project...
Damage model loaded.


In [9]:
def detect_damage(image_path):
    result = damage_model_rf.predict(image_path, confidence=int(DAMAGE_CONFIDENCE_THRESHOLD * 100), overlap=30).json()
    detections = result.get("predictions", [])
    parsed = [{"class": d.get("class"), "confidence": d.get("confidence", 0.0)} for d in detections]
    max_confidence = max((d["confidence"] for d in parsed), default=0.0)
    is_damaged = len(parsed) > 0
    return is_damaged, max_confidence, parsed


## 7. Load the traffic sign model (GitHub)

In [10]:
if not os.path.exists(SIGN_REPO_LOCAL_DIR):
    os.system(f"git clone {SIGN_REPO_URL} {SIGN_REPO_LOCAL_DIR}")

model_dir = os.path.join(SIGN_REPO_LOCAL_DIR, "model")
print("Files in the model/ folder:")
for f in os.listdir(model_dir):
    print(" ", f)


Files in the model/ folder:
  traffic_sign_detector.pt


In [11]:
sign_weights_path = os.path.join(model_dir, SIGN_WEIGHTS_FILENAME)
sign_model = YOLO(sign_weights_path)
print("Traffic sign model loaded. Classes:", sign_model.names)


Traffic sign model loaded. Classes: {0: 'Green Light', 1: 'Red Light', 2: 'Speed Limit 10', 3: 'Speed Limit 100', 4: 'Speed Limit 110', 5: 'Speed Limit 120', 6: 'Speed Limit 20', 7: 'Speed Limit 30', 8: 'Speed Limit 40', 9: 'Speed Limit 50', 10: 'Speed Limit 60', 11: 'Speed Limit 70', 12: 'Speed Limit 80', 13: 'Speed Limit 90', 14: 'Stop'}


In [12]:
def detect_traffic_signs(image_path):
    results = sign_model.predict(image_path, verbose=False)
    detections = []
    for r in results:
        for box in r.boxes:
            conf = float(box.conf[0])
            cls_name = sign_model.names[int(box.cls[0])]
            if cls_name in IGNORED_SIGN_CLASSES or conf < SIGN_CONFIDENCE_THRESHOLD:
                continue
            detections.append({"class": cls_name, "confidence": conf})
    return detections


## 7b. Speed bump detector (your trained model)

In [13]:
if not os.path.exists(SPEED_BUMP_MODEL_PATH):
    raise FileNotFoundError(
        f"{SPEED_BUMP_MODEL_PATH} not found. Download it from your Kaggle notebook's Output tab "
        f"and place it next to this notebook."
    )

bump_model = YOLO(SPEED_BUMP_MODEL_PATH)
print("Speed bump detector loaded. Classes:", bump_model.names)


Speed bump detector loaded. Classes: {0: 'Rumble Strip', 1: 'Speed-Bump'}


In [14]:
def detect_speed_bump(image_path):
    """
    Returns (bump_ahead: bool, max_confidence: float, detections: list).
    Only the 'Speed-Bump' class triggers a slowdown — 'Rumble Strip' is detected and returned for
    visibility but doesn't currently change speed on its own.
    """
    results = bump_model.predict(image_path, conf=SPEED_BUMP_CONFIDENCE_THRESHOLD, verbose=False)
    detections = []
    for r in results:
        for box in r.boxes:
            conf = float(box.conf[0])
            cls_name = bump_model.names[int(box.cls[0])]
            detections.append({"class": cls_name, "confidence": conf})

    bump_detections = [d for d in detections if d["class"] == "Speed-Bump"]
    bump_ahead = len(bump_detections) > 0
    max_confidence = max((d["confidence"] for d in bump_detections), default=0.0)

    return bump_ahead, max_confidence, detections


## 7c. Pedestrian + crosswalk detector (your trained model)
*"pedestrian ahead" is currently just "a person was confidently detected anywhere in frame" — it doesn't yet check whether they're actually in the car's path. Good enough for an MVP hard-stop trigger, worth refining later.*

In [15]:
if not os.path.exists(PED_CROSSWALK_MODEL_PATH):
    raise FileNotFoundError(
        f"{PED_CROSSWALK_MODEL_PATH} not found. Download it from your Kaggle notebook's Output tab "
        f"and place it next to this notebook."
    )

ped_crosswalk_model = YOLO(PED_CROSSWALK_MODEL_PATH)
print("Pedestrian + crosswalk detector loaded. Classes:", ped_crosswalk_model.names)


Pedestrian + crosswalk detector loaded. Classes: {0: 'person', 1: 'cross walk'}


In [16]:
def detect_pedestrian_crosswalk(image_path):
    """
    Returns (pedestrian_close: bool, pedestrian_far: bool, crosswalk_ahead: bool, detections: list).
    A detected person only counts as "close" (mandatory stop) if their box is large/low in frame —
    same distance heuristic as the vehicle safety-distance check. A person detected but NOT close is
    "far" — advisory slow-down suggestion instead of a hard stop.
    """
    img = Image.open(image_path)
    frame_w, frame_h = img.size

    results = ped_crosswalk_model.predict(
        image_path, conf=min(PEDESTRIAN_CONFIDENCE_THRESHOLD, CROSSWALK_CONFIDENCE_THRESHOLD), verbose=False
    )
    detections = []
    for r in results:
        for box in r.boxes:
            conf = float(box.conf[0])
            cls_name = ped_crosswalk_model.names[int(box.cls[0])]
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            detections.append({
                "class": cls_name,
                "confidence": conf,
                "relative_height": (y2 - y1) / frame_h,
                "relative_bottom": y2 / frame_h,
            })

    pedestrian_detections = [d for d in detections if d["class"] == "person" and d["confidence"] >= PEDESTRIAN_CONFIDENCE_THRESHOLD]
    crosswalk_detections = [d for d in detections if d["class"] == "cross walk" and d["confidence"] >= CROSSWALK_CONFIDENCE_THRESHOLD]

    close_pedestrians = [
        d for d in pedestrian_detections
        if d["relative_height"] >= PEDESTRIAN_CLOSE_RELATIVE_HEIGHT_THRESHOLD
        or d["relative_bottom"] >= PEDESTRIAN_CLOSE_RELATIVE_BOTTOM_THRESHOLD
    ]

    pedestrian_close = len(close_pedestrians) > 0
    pedestrian_far = len(pedestrian_detections) > 0 and not pedestrian_close
    crosswalk_ahead = len(crosswalk_detections) > 0

    return pedestrian_close, pedestrian_far, crosswalk_ahead, detections


## 7d. Weather classifier (your trained model, .tflite)

In [17]:
if not os.path.exists(WEATHER_MODEL_PATH):
    raise FileNotFoundError(
        f"{WEATHER_MODEL_PATH} not found. Download it from your Kaggle notebook's Output tab "
        f"and place it next to this notebook."
    )

weather_interpreter = tf.lite.Interpreter(model_path=WEATHER_MODEL_PATH)
weather_interpreter.allocate_tensors()
weather_input_details = weather_interpreter.get_input_details()
weather_output_details = weather_interpreter.get_output_details()
print("Weather classifier loaded.")


Weather classifier loaded.


    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    


In [19]:
def detect_weather(image_path):
    img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0).astype(weather_input_details[0]["dtype"])

    weather_interpreter.set_tensor(weather_input_details[0]["index"], img_array)
    weather_interpreter.invoke()
    predictions = weather_interpreter.get_tensor(weather_output_details[0]["index"])[0]

    predicted_class = WEATHER_CLASSES[np.argmax(predictions)]
    confidence = float(np.max(predictions))

    if predicted_class == "adverse" and confidence < WEATHER_CONFIDENCE_THRESHOLD:
        return "clear", confidence  # not confident enough to trust the adverse call

    return predicted_class, confidence


## 7e. Vehicle detector for safety distance (stock YOLOv8n)
Heuristic only, not a real metric distance — a vehicle whose box is tall relative to the frame, or whose bottom edge sits low in frame, is considered close. Calibrate the two thresholds in the config cell against your own sample frames before trusting this.

In [ ]:
vehicle_model = YOLO("yolov8n.pt")  # stock COCO weights, auto-downloads on first run
print("Vehicle detector loaded (stock YOLOv8n). Classes:", vehicle_model.names)


In [20]:
def detect_car_too_close(image_path):
    """
    Returns (too_close: bool, closest_vehicle: dict or None, detections: list).
    """
    img = Image.open(image_path)
    frame_w, frame_h = img.size

    results = vehicle_model.predict(image_path, conf=VEHICLE_CONFIDENCE_THRESHOLD, verbose=False)
    detections = []

    for r in results:
        for box in r.boxes:
            cls_name = vehicle_model.names[int(box.cls[0])]
            if cls_name not in VEHICLE_CLASSES:
                continue
            conf = float(box.conf[0])
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            relative_height = (y2 - y1) / frame_h
            relative_bottom = y2 / frame_h

            detections.append({
                "class": cls_name,
                "confidence": conf,
                "relative_height": relative_height,
                "relative_bottom": relative_bottom,
            })

    close_vehicles = [
        d for d in detections
        if d["relative_height"] >= CLOSE_RELATIVE_HEIGHT_THRESHOLD
        or d["relative_bottom"] >= CLOSE_RELATIVE_BOTTOM_THRESHOLD
    ]

    too_close = len(close_vehicles) > 0
    closest = max(close_vehicles, key=lambda d: d["relative_bottom"], default=None)

    return too_close, closest, detections


## 8. Speed logic — base speed + reason-code overrides
Priority, highest to lowest: pedestrian close → stop / red light → crosswalk → speed bump → posted speed limit → car too close → adverse weather. The first two are automatic (no confirmation, see `AUTOMATIC_REASON_CODES`); everything after is a confirmable suggestion — see the Voice Assistant section.

In [22]:
BASE_SPEED_BY_ROAD_TYPE = {
    "autoroute": 120,
    "urbaine": 60,
    "rurale": 100,
    "off-road": 50,  # tuned down from 70 -- matches the cloud pipeline's latest calibration
}
DAMAGE_SPEED_REDUCTION = 0.30

CROSSWALK_SPEED_CAP = 30
SPEED_BUMP_SPEED_CAP = 30
SAFETY_DISTANCE_SPEED_REDUCTION = 0.20
WEATHER_SPEED_REDUCTION = 0.20
PEDESTRIAN_FAR_SPEED_REDUCTION = 0.30

# Reason codes that bypass voice confirmation entirely -- genuine emergency stops, where waiting
# on a driver's spoken "yes" before braking would defeat the point. Everything else becomes a
# confirmable suggestion (see the Voice Assistant / Speed-change announcements sections below).
AUTOMATIC_REASON_CODES = {"pedestrian_stop", "stop_sign", "red_light"}


def get_base_speed(road_type, is_damaged):
    speed = BASE_SPEED_BY_ROAD_TYPE[road_type]
    if is_damaged:
        speed = speed * (1 - DAMAGE_SPEED_REDUCTION)
    return round(speed, 1)


def apply_overrides(
    current_speed,
    detected_signs,
    pedestrian_close=False,
    pedestrian_far=False,
    crosswalk_ahead=False,
    bump_ahead=False,
    car_too_close=False,
    weather_condition="clear",
):
    """
    Returns (final_speed, reason_code, reason_details). reason_code is one of:
    "pedestrian_stop", "stop_sign", "red_light", "crosswalk", "speed_bump", "speed_limit",
    "pedestrian_far", "safety_distance", "weather", or None.
    """
    if pedestrian_close:
        return 0, "pedestrian_stop", {}

    for sign in detected_signs:
        if sign["class"] == "Stop":
            return 0, "stop_sign", {}
        if sign["class"] == "Red Light":
            return 0, "red_light", {}
    # NOTE: Yellow/amber light is NOT a class this model detects.

    final_speed = current_speed
    reason_code = None
    reason_details = {}

    if crosswalk_ahead and CROSSWALK_SPEED_CAP < final_speed:
        final_speed = CROSSWALK_SPEED_CAP
        reason_code, reason_details = "crosswalk", {"speed": CROSSWALK_SPEED_CAP}

    if bump_ahead and SPEED_BUMP_SPEED_CAP < final_speed:
        final_speed = SPEED_BUMP_SPEED_CAP
        reason_code, reason_details = "speed_bump", {"speed": SPEED_BUMP_SPEED_CAP}

    for sign in detected_signs:
        match = re.search(r"Speed Limit (\d+)", sign["class"])
        if match:
            posted_limit = int(match.group(1))
            if posted_limit < final_speed:
                final_speed = posted_limit
                reason_code, reason_details = "speed_limit", {"speed": posted_limit}

    if pedestrian_far:
        final_speed = round(final_speed * (1 - PEDESTRIAN_FAR_SPEED_REDUCTION), 1)
        reason_code, reason_details = "pedestrian_far", {"speed": final_speed}

    if car_too_close:
        final_speed = round(final_speed * (1 - SAFETY_DISTANCE_SPEED_REDUCTION), 1)
        reason_code, reason_details = "safety_distance", {"speed": final_speed}

    if weather_condition == "adverse":
        final_speed = round(final_speed * (1 - WEATHER_SPEED_REDUCTION), 1)
        reason_code, reason_details = "weather", {"speed": final_speed}

    return final_speed, reason_code, reason_details


## 8b. OpenStreetMap lookup (Overpass API) — the primary source, camera models are the fallback
One Overpass call per (rounded) GPS location, cached so a slow-moving or stationary vehicle doesn't re-query on every single frame. Returns `None`/`False` for anything OSM has no tag for at this spot — `classify_full()` below falls back to the matching camera model only for those specific gaps, not for everything.

In [ ]:
_overpass_cache = {}


def _overpass_query(latitude, longitude):
    cache_key = (round(latitude, OVERPASS_CACHE_PRECISION), round(longitude, OVERPASS_CACHE_PRECISION))
    if cache_key in _overpass_cache:
        return _overpass_cache[cache_key]

    query = f"""
    [out:json][timeout:{OVERPASS_TIMEOUT_SECONDS}];
    (
      way(around:{OVERPASS_SEARCH_RADIUS_M},{latitude},{longitude})["highway"];
      node(around:{OVERPASS_FEATURE_RADIUS_M},{latitude},{longitude})["highway"~"^(stop|traffic_signals|crossing)$"];
      node(around:{OVERPASS_FEATURE_RADIUS_M},{latitude},{longitude})["traffic_calming"];
    );
    out body;
    """

    try:
        response = requests.get(OVERPASS_URL, params={"data": query}, timeout=OVERPASS_TIMEOUT_SECONDS)
        response.raise_for_status()
        elements = response.json().get("elements", [])
    except Exception as e:
        print(f"Overpass query failed ({e}) -- falling back to camera models for everything at this location.")
        elements = []

    _overpass_cache[cache_key] = elements
    return elements


def get_overpass_context(latitude, longitude):
    """
    Returns what OSM knows about this exact spot. Any field left as None/False means Overpass had
    no answer -- classify_full() falls back to the matching camera model for that field only.
    """
    elements = _overpass_query(latitude, longitude)

    context = {
        "road_type": None,
        "is_damaged": None,
        "posted_speed_limit_kmh": None,
        "stop_sign_present": False,
        "crosswalk_ahead": False,
        "bump_ahead": False,
    }

    ways = [e for e in elements if e.get("type") == "way" and "tags" in e]
    if ways:
        # First returned way stands in for "the road we're on" -- Overpass doesn't sort by
        # distance to the query point, but with a tight search radius this is a fair approximation.
        tags = ways[0]["tags"]
        highway = tags.get("highway")
        surface = tags.get("surface")

        if surface in UNPAVED_SURFACE_TAGS:
            context["road_type"] = "off-road"
        elif highway in AUTOROUTE_HIGHWAY_TAGS:
            context["road_type"] = "autoroute"
        elif highway in URBAINE_HIGHWAY_TAGS:
            context["road_type"] = "urbaine"
        elif highway in RURALE_HIGHWAY_TAGS:
            context["road_type"] = "rurale"
        # else: leave as None -- unrecognized/missing highway tag, fall back to the camera classifier

        smoothness = tags.get("smoothness")
        if smoothness in BAD_SMOOTHNESS_TAGS:
            context["is_damaged"] = True
        elif smoothness in GOOD_SMOOTHNESS_TAGS:
            context["is_damaged"] = False
        # else: leave as None -- no smoothness tag, fall back to the damage model

        maxspeed = tags.get("maxspeed")
        if maxspeed and maxspeed.split()[0].isdigit():
            context["posted_speed_limit_kmh"] = int(maxspeed.split()[0])
        # else: leave as None -- no maxspeed tag, fall back to sign detection

    for e in elements:
        if e.get("type") != "node" or "tags" not in e:
            continue
        tags = e["tags"]
        if tags.get("highway") == "stop":
            context["stop_sign_present"] = True
        if tags.get("highway") == "crossing":
            context["crosswalk_ahead"] = True
        if "traffic_calming" in tags:
            context["bump_ahead"] = True

    return context


# Bounding boxes (south, north, west, east) for areas with dense OSM road tagging AND solid
# Street View coverage -- used to generate a random, but genuinely WORKING, test location each
# run, instead of relying on laptop IP geolocation (fixed, inaccurate, not what testing needs).
TEST_AREAS = {
    "Casablanca":            (33.55, 33.60, -7.65, -7.58),
    "Rabat":                 (34.00, 34.03, -6.85, -6.80),
    "Paris":                 (48.845, 48.875, 2.30, 2.40),
    "London":                (51.49, 51.52, -0.15, -0.05),
    "New York (Manhattan)":  (40.745, 40.775, -74.00, -73.96),
}


def get_random_test_location(max_attempts=15):
    """
    Picks a random point inside one of TEST_AREAS and confirms it actually has Street View
    coverage (via the metadata endpoint) before returning it -- retries with a new random point
    up to max_attempts times, so a run never hands fetch_road_image() a dead coordinate.
    """
    for attempt in range(max_attempts):
        area_name = random.choice(list(TEST_AREAS.keys()))
        south, north, west, east = TEST_AREAS[area_name]
        latitude = random.uniform(south, north)
        longitude = random.uniform(west, east)

        metadata = requests.get(STREET_VIEW_METADATA_URL, params={
            "location": f"{latitude},{longitude}",
            "key": GOOGLE_MAPS_API_KEY,
        }, timeout=10).json()

        if metadata.get("status") == "OK":
            print(f"Test location: {area_name} ({latitude:.5f}, {longitude:.5f}) -- Street View confirmed.")
            return latitude, longitude

    raise RuntimeError(
        f"Couldn't find a Street-View-covered point after {max_attempts} attempts -- "
        f"try again, or widen/add to TEST_AREAS."
    )


def get_overpass_info():
    """
    Function 1 of 3: picks a random, Street-View-confirmed test location (see
    get_random_test_location() above) and returns everything OpenStreetMap knows at that spot.
    Takes no arguments -- a fresh, different real-world location is chosen every call, which is
    the point of a test iteration, rather than the same fixed point every time.
    """
    latitude, longitude = get_random_test_location()
    context = get_overpass_context(latitude, longitude)
    context["latitude"] = latitude
    context["longitude"] = longitude
    return context


## 8c. Fetch a road-level image from Street View, by coordinates
**Not a live camera feed** — Street View photos are static and can be months or years old, and coverage has real gaps (rural roads, private roads, many non-Western countries). `fetch_road_image` returns `None` when there's no coverage, so the code below falls back to your existing manual file picker instead of silently failing.

In [ ]:
def fetch_road_image(latitude, longitude, output_path="street_view_frame.jpg", heading=0, fov=90, pitch=0):
    """
    Function 2 of 3: pulls a forward-facing Street View photo at these coordinates as a stand-in
    for a live camera frame. Checks coverage via the metadata endpoint first, so we fail cleanly
    (return None) instead of saving a "no imagery available" placeholder as if it were real.
    """
    metadata = requests.get(STREET_VIEW_METADATA_URL, params={
        "location": f"{latitude},{longitude}",
        "key": GOOGLE_MAPS_API_KEY,
    }, timeout=10).json()

    if metadata.get("status") != "OK":
        print(f"No Street View coverage at ({latitude}, {longitude}): {metadata.get('status')}")
        return None

    response = requests.get(STREET_VIEW_URL, params={
        "size": "640x640",
        "location": f"{latitude},{longitude}",
        "heading": heading,
        "fov": fov,
        "pitch": pitch,
        "key": GOOGLE_MAPS_API_KEY,
    }, timeout=10)
    response.raise_for_status()

    with open(output_path, "wb") as f:
        f.write(response.content)

    return output_path


## 9. Camera-model execution — fills whatever Overpass couldn't answer

In [ ]:
def run_camera_models(image_path, overpass_info):
    """
    Function 3 of 3: runs only the camera models needed to fill gaps in overpass_info (whatever
    OSM didn't answer for this location), plus the four fields that are always camera-based
    regardless -- red light state, pedestrian presence, safety distance, weather. Combines with
    the existing speed logic exactly as before. This is the old classify_full(), split into three
    pieces: coordinate + Overpass lookup (get_overpass_info), image acquisition
    (fetch_road_image / manual picker), and camera-model execution (this function).
    """
    if overpass_info["road_type"] is not None:
        road_type = overpass_info["road_type"]
        road_type_source = "overpass"
    else:
        is_offroad, offroad_conf = detect_offroad(image_path)
        if is_offroad:
            road_type = "off-road"
        else:
            img = tf.keras.utils.load_img(image_path, target_size=IMG_SIZE)
            img_array = tf.keras.utils.img_to_array(img)
            img_array = tf.expand_dims(img_array, 0)
            predictions = road_classifier.predict(img_array, verbose=0)
            road_type = CLASS_NAMES[np.argmax(predictions[0])]
        road_type_source = "camera"

    if overpass_info["is_damaged"] is not None:
        is_damaged = overpass_info["is_damaged"]
        damage_source = "overpass"
    else:
        is_damaged, damage_conf, damage_detections = detect_damage(image_path)
        damage_source = "camera"

    if overpass_info["bump_ahead"]:
        bump_ahead = True
        bump_source = "overpass"
    else:
        bump_ahead, bump_conf, bump_detections = detect_speed_bump(image_path)
        bump_source = "camera"

    detected_signs = detect_traffic_signs(image_path)  # needed for red light state regardless
    pedestrian_close, pedestrian_far, crosswalk_ahead_camera, ped_detections = detect_pedestrian_crosswalk(image_path)
    car_too_close, closest_vehicle, vehicle_detections = detect_car_too_close(image_path)
    weather_condition, weather_conf = detect_weather(image_path)

    crosswalk_ahead = overpass_info["crosswalk_ahead"] or crosswalk_ahead_camera
    crosswalk_source = "overpass" if overpass_info["crosswalk_ahead"] else "camera"

    if overpass_info["stop_sign_present"]:
        detected_signs = detected_signs + [{"class": "Stop", "confidence": 1.0, "source": "overpass"}]

    if overpass_info["posted_speed_limit_kmh"] is not None:
        detected_signs = [s for s in detected_signs if not re.match(r"Speed Limit \d+", s["class"])]
        detected_signs = detected_signs + [{
            "class": f"Speed Limit {overpass_info['posted_speed_limit_kmh']}",
            "confidence": 1.0,
            "source": "overpass",
        }]

    base_speed = get_base_speed(road_type, is_damaged)
    final_speed, reason_code, reason_details = apply_overrides(
        base_speed,
        detected_signs,
        pedestrian_close=pedestrian_close,
        pedestrian_far=pedestrian_far,
        crosswalk_ahead=crosswalk_ahead,
        bump_ahead=bump_ahead,
        car_too_close=car_too_close,
        weather_condition=weather_condition,
    )

    return {
        "road_type": road_type,
        "damaged": is_damaged,
        "detected_signs": detected_signs,
        "pedestrian_close": pedestrian_close,
        "pedestrian_far": pedestrian_far,
        "crosswalk_ahead": crosswalk_ahead,
        "speed_bump_detected": bump_ahead,
        "car_too_close": car_too_close,
        "weather_condition": weather_condition,
        "base_speed": base_speed,
        "final_speed_kmh": final_speed,
        "reason_code": reason_code,
        "reason_details": reason_details,
        "info_sources": {
            "road_type": road_type_source,
            "damaged": damage_source,
            "speed_bump_detected": bump_source,
            "crosswalk_ahead": crosswalk_source,
            "stop_sign": "overpass" if overpass_info["stop_sign_present"] else "camera",
            "speed_limit": "overpass" if overpass_info["posted_speed_limit_kmh"] is not None else "camera",
            "red_light": "camera",       # always -- no OSM equivalent
            "pedestrian": "camera",      # always
            "safety_distance": "camera", # always
            "weather": "camera",         # always
        },
    }


## 10. Run it: auto-coordinates → Street View image → camera models for the gaps

In [ ]:
def pick_image_file():
    try:
        root = tk.Tk()
        root.withdraw()
        root.attributes('-topmost', True)
        root.lift()
        root.focus_force()
        file_path = filedialog.askopenfilename(
            parent=root,
            title="Select a road image",
            filetypes=[("Image files", "*.jpg *.jpeg *.png")]
        )
        root.destroy()
        return file_path
    except Exception as e:
        print(f"Tkinter dialog failed: {e}")
        path = input("Enter the full path to the image file: ").strip().strip('"')
        return path if os.path.exists(path) else None


# --- Orchestration: function 1 (coordinates + Overpass) -> function 2 (Street View image, with a
# manual-picker fallback for when there's no coverage) -> function 3 (camera models for the gaps) ---
overpass_info = get_overpass_info()
print(f"Location (IP-based, approximate): {overpass_info['latitude']}, {overpass_info['longitude']}")

image_path = fetch_road_image(overpass_info["latitude"], overpass_info["longitude"])

if image_path is None:
    print("No Street View coverage here -- falling back to picking a local image file instead.")
    image_path = pick_image_file()

if image_path:
    result = run_camera_models(image_path, overpass_info)

    img = Image.open(image_path)
    plt.imshow(img)
    plt.axis("off")
    title = (
        f"{result['road_type']} | damaged: {result['damaged']} | "
        f"ped_close: {result['pedestrian_close']} | ped_far: {result['pedestrian_far']} | "
        f"crosswalk: {result['crosswalk_ahead']} | bump: {result['speed_bump_detected']} | "
        f"close: {result['car_too_close']} | weather: {result['weather_condition']} | "
        f"speed: {result['final_speed_kmh']} km/h"
    )
    if result["reason_code"]:
        title += f"\n({result['reason_code']})"
    plt.title(title)
    plt.show()

    print(result)
    print("\nInfo sources (overpass vs camera):")
    for field, source in result["info_sources"].items():
        print(f"  {field}: {source}")
else:
    print("No image available (no Street View coverage and no file selected).")


## 11. Voice assistant — intent understanding, propose→confirm, ElevenLabs TTS
Nothing here executes on its own anymore. `understand_and_respond()` only ever *proposes* a command and asks the driver a confirmation question; `dispatch_command()` is only ever called from `confirm_pending_command()`, after the driver's next reply is judged as a yes. Small talk and questions still answer directly, with no confirmation step — only real vehicle commands go through the propose→confirm flow. Also keeps a rolling text-only history of the last `CONVERSATION_HISTORY_TURNS` exchanges (transcripts, not audio) and feeds it back in on every call, so the assistant has continuity across turns ("what did I just ask you?").

In [ ]:
whisper_model = whisper.load_model("base")

def transcribe_audio(audio_path):
    """
    No language= argument -> Whisper auto-detects the spoken language.
    Returns (transcript, detected_language_code), e.g. ("hello there", "en").
    """
    result = whisper_model.transcribe(audio_path)
    return result["text"].strip(), result["language"]


import json
from groq import Groq


groq_client = Groq(api_key=GROQ_API_KEY)
LLM_MODEL = "openai/gpt-oss-120b"


# --- Vehicle command tool: OpenAI-style tool schema (Groq uses the OpenAI-compatible API) ---
VEHICLE_TOOL = {
    "type": "function",
    "function": {
        "name": "vehicle_command",
        "description": (
            "Call this when the user gives an instruction that changes something "
            "about the car (window, speed, cruise speed, eco mode, safety distance). "
            "Do NOT call this for questions, greetings, or general conversation."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "action": {
                    "type": "string",
                    "enum": [
                        "close_window", "open_window",
                        "set_speed_limit", "set_cruise_speed",
                        "enable_eco_mode",
                        "increase_safety_distance", "decrease_safety_distance",
                        "decrease_speed", "increase_speed",
                    ],
                },
                "value": {
                    "type": "integer",
                    "description": "Speed in km/h. Only used for set_speed_limit and set_cruise_speed.",
                },
            },
            "required": ["action"],
        },
    },
}

# --- Confirmation tool: judges the driver's yes/no reply, in whatever language they used ---
CONFIRMATION_TOOL = {
    "type": "function",
    "function": {
        "name": "record_confirmation",
        "description": "Call this to record whether the driver confirmed or declined the pending action.",
        "parameters": {
            "type": "object",
            "properties": {
                "confirmed": {
                    "type": "boolean",
                    "description": "true if the driver agreed/said yes, false if they declined/said no.",
                },
            },
            "required": ["confirmed"],
        },
    },
}

SYSTEM_PROMPT = """You are the voice assistant embedded in an adaptive cruise control (ADAS) system in a car.

Always reply in the exact same language the user just spoke to you in, whatever language that is.

You have two jobs:
1. If the user gives a command to change something about the car (window, speed limit, cruise speed, eco mode, safety distance, speed up/slow down), call the vehicle_command tool with the right action (and value in km/h if relevant). IMPORTANT: nothing is applied yet at this point -- along with the tool call, phrase your spoken reply as a short CONFIRMATION QUESTION asking the driver if they want you to do it (e.g. "Do you want me to set cruise speed to 100?"), in the same language the user spoke. The action only actually happens after the driver confirms on the next turn.
2. For anything else -- greetings, small talk, "who are you", "what do you do", general questions -- just answer normally and conversationally, like any helpful voice assistant would. Keep replies short (1-3 sentences), since this is spoken out loud in a car, not read on a screen.

Only call the tool for real commands. Never invent vehicle behavior you were not asked for.
"""

_pending_command = {"data": None, "question": None}  # command awaiting driver confirmation

# --- Conversation history: last ~5 exchanges, transcribed text only (no audio kept) ---
CONVERSATION_HISTORY_TURNS = 5  # how many back-and-forth exchanges to remember

_conversation_history = []  # flat list of {"role": "user"/"assistant", "text": ...}, oldest first


def _append_history(user_text, assistant_text):
    _conversation_history.append({"role": "user", "text": user_text})
    _conversation_history.append({"role": "assistant", "text": assistant_text})
    max_entries = CONVERSATION_HISTORY_TURNS * 2
    if len(_conversation_history) > max_entries:
        del _conversation_history[: len(_conversation_history) - max_entries]


def _history_as_messages():
    return [{"role": turn["role"], "content": turn["text"]} for turn in _conversation_history]


def dispatch_command(action, value, vehicle_state):
    if action == "set_speed_limit" and value is not None:
        vehicle_state["speed_limit_kmh"] = value
    elif action == "set_cruise_speed" and value is not None:
        vehicle_state["cruise_speed_kmh"] = value
    elif action == "enable_eco_mode":
        vehicle_state["eco_mode"] = True
    elif action == "increase_safety_distance":
        vehicle_state["safety_distance_level"] += 1
    elif action == "decrease_safety_distance":
        vehicle_state["safety_distance_level"] = max(1, vehicle_state["safety_distance_level"] - 1)
    elif action == "decrease_speed":
        vehicle_state["cruise_speed_kmh"] = max(0, vehicle_state["cruise_speed_kmh"] - 10)
    elif action == "increase_speed":
        vehicle_state["cruise_speed_kmh"] += 10
    elif action == "close_window":
        vehicle_state["window_open"] = False
    elif action == "open_window":
        vehicle_state["window_open"] = True
    return vehicle_state


def _judge_yes_no(confirmation_text, question_asked):
    """
    Language-agnostic yes/no judgment for a driver's spoken reply to a confirmation question --
    uses Groq instead of a fixed keyword list, since the driver may reply in any language.
    Returns (confirmed: bool, spoken_reply: str).
    """
    response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    f'You are the voice assistant in a car. You just asked the driver: "{question_asked}"\n'
                    "The driver has now replied. Call record_confirmation with confirmed=true if they "
                    "agreed/said yes, confirmed=false if they declined/said no.\n"
                    "Also write one short spoken sentence in the same language the driver just used: "
                    "if confirmed, say you're doing it now; if declined, acknowledge you won't change anything."
                ),
            },
            {"role": "user", "content": confirmation_text},
        ],
        tools=[CONFIRMATION_TOOL],
        tool_choice="auto",
    )

    message = response.choices[0].message

    confirmed = False
    if message.tool_calls:
        for call in message.tool_calls:
            if call.function.name == "record_confirmation":
                args = json.loads(call.function.arguments)
                confirmed = bool(args.get("confirmed"))

    spoken_reply = (message.content or "").strip()
    if not spoken_reply:
        spoken_reply = "Done." if confirmed else "Okay, no changes made."

    return confirmed, spoken_reply


def understand_and_respond(user_text, vehicle_state):
    """
    Phase 1 (propose): either answers small talk directly, or identifies a real vehicle command
    and asks the driver a confirmation question -- without touching vehicle_state yet.
    Returns (spoken_text, vehicle_state, needs_confirmation).
    """
    messages = (
        [{"role": "system", "content": SYSTEM_PROMPT}]
        + _history_as_messages()
        + [{"role": "user", "content": user_text}]
    )

    response = groq_client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        tools=[VEHICLE_TOOL],
        tool_choice="auto",
    )

    message = response.choices[0].message

    pending_action = None
    if message.tool_calls:
        for call in message.tool_calls:
            if call.function.name == "vehicle_command":
                args = json.loads(call.function.arguments)
                pending_action = {"action": args.get("action"), "value": args.get("value")}

    spoken_text = (message.content or "").strip()
    if not spoken_text:
        spoken_text = "Done." if pending_action is None else "Do you confirm?"

    if pending_action is not None:
        _pending_command["data"] = pending_action
        _pending_command["question"] = spoken_text

    _append_history(user_text, spoken_text)

    return spoken_text, vehicle_state, pending_action is not None


def confirm_pending_command(confirmation_text, vehicle_state):
    """
    Phase 2 (confirm): driver's spoken yes/no in -> applies the pending command (or cancels it).
    Returns (updated_vehicle_state, response_text).
    """
    pending = _pending_command["data"]
    if pending is None:
        return vehicle_state, "There is no pending command to confirm."

    confirmed, spoken_reply = _judge_yes_no(confirmation_text, _pending_command["question"])
    if confirmed:
        vehicle_state = dispatch_command(pending["action"], pending["value"], vehicle_state)

    _append_history(confirmation_text, spoken_reply)

    _pending_command["data"] = None
    _pending_command["question"] = None
    return vehicle_state, spoken_reply


elevenlabs_client = ElevenLabs(api_key=ELEVENLABS_API_KEY)

def speak(text, output_path="response.mp3", voice_id=ELEVENLABS_VOICE_ID, language_code="en"):
    audio = elevenlabs_client.text_to_speech.convert(
        text=text,
        voice_id=voice_id,
        model_id=ELEVENLABS_MODEL_ID,
        language_code=language_code,
        output_format="mp3_44100_128",
    )
    save_audio(audio, output_path)
    return output_path


from pydub import AudioSegment
import numpy as np

def play_audio_blocking(path):
    """
    Plays an mp3 and blocks until it's ACTUALLY finished -- replaces the old
    autoplay + guessed-duration approach, which let the mic start recording
    too early and pick up the assistant's own tail end.
    """
    audio_segment = AudioSegment.from_mp3(path)
    samples = np.array(audio_segment.get_array_of_samples())
    if audio_segment.channels == 2:
        samples = samples.reshape((-1, 2))
    sd.play(samples, samplerate=audio_segment.frame_rate)
    sd.wait()  # genuinely blocks until playback ends, no guessing

## 12. Record your own voice with your laptop's microphone (propose→confirm)

In [ ]:
def record_audio(duration=4, samplerate=16000, output_path="mic_recording.wav"):
    print(f"Recording for {duration} seconds -- speak now, any language...")
    recording = sd.rec(int(duration * samplerate), samplerate=samplerate, channels=1, dtype='int16')
    sd.wait()
    write_wav(output_path, samplerate, recording)
    return output_path


vehicle_state = {
    "cruise_speed_kmh": 50,
    "speed_limit_kmh": None,
    "eco_mode": False,
    "safety_distance_level": 2,
    "window_open": True,
}

print("Starting conversation -- stop the cell (Interrupt/Stop button) to end it.\n")

try:
    while True:
        # --- Phase 1: propose ---
        audio_path = record_audio(duration=4)
        text, detected_language = transcribe_audio(audio_path)

        if not text:
            continue  # caught silence, just listen again

        print(f"Whisper heard ({detected_language}): {text!r}")

        response_text, vehicle_state, needs_confirmation = understand_and_respond(text, vehicle_state)
        print(f"Assistant: {response_text}")

        response_audio = speak(response_text, language_code=detected_language)
        play_audio_blocking(response_audio)

        # --- Phase 2: confirm (only if a real command was proposed) ---
        if needs_confirmation:
            print("Waiting for your confirmation...")
            confirm_audio_path = record_audio(duration=4, output_path="confirmation.wav")
            confirmation_text, confirm_language = transcribe_audio(confirm_audio_path)
            print(f"Whisper heard ({confirm_language}): {confirmation_text!r}")

            vehicle_state, confirm_response = confirm_pending_command(confirmation_text, vehicle_state)
            print(f"Assistant: {confirm_response}")

            confirm_audio = speak(confirm_response, language_code=confirm_language)
            play_audio_blocking(confirm_audio)

        print(f"Vehicle state: {vehicle_state}\n")

except KeyboardInterrupt:
    print("Conversation stopped.")

## 13. Speed-change announcements (reason-code driven, ties the image pipeline to the voice assistant)
`AUTOMATIC_REASON_CODES` (pedestrian_stop, stop_sign, red_light) are genuine emergency stops: announced immediately as a fact, applied immediately, no confirmation asked -- waiting on a spoken "yes" before braking for a pedestrian would defeat the point. Everything else is a SUGGESTION: announced as a question and held as pending, only applied once `confirm_pending_suggestion()` sees a "yes" -- same propose→confirm pattern as the voice-command flow above, just triggered by the camera instead of the driver's speech.

These announcements speak in a fixed `ANNOUNCEMENT_LANGUAGE` rather than the driver's detected language, since they're not a reply to something the driver said -- there's no detected language to match here.

In [ ]:
ANNOUNCEMENT_LANGUAGE = "fr"

SUGGESTION_QUESTION_TEMPLATES = {
    "crosswalk":       "Passage pi\u00e9ton d\u00e9tect\u00e9. Voulez-vous ralentir \u00e0 {speed} kilom\u00e8tres heure ?",
    "speed_bump":      "Dos d'\u00e2ne d\u00e9tect\u00e9. Voulez-vous ralentir \u00e0 {speed} kilom\u00e8tres heure ?",
    "speed_limit":     "Limite de vitesse affich\u00e9e : {speed} kilom\u00e8tres heure. Voulez-vous l'appliquer ?",
    "pedestrian_far":  "Pi\u00e9ton rep\u00e9r\u00e9 au loin. Voulez-vous ralentir \u00e0 {speed} kilom\u00e8tres heure ?",
    "safety_distance": "V\u00e9hicule proche devant. Voulez-vous ralentir \u00e0 {speed} kilom\u00e8tres heure ?",
    "weather":         "Conditions m\u00e9t\u00e9o d\u00e9grad\u00e9es. Voulez-vous ralentir \u00e0 {speed} kilom\u00e8tres heure ?",
    None:              "Vitesse sugg\u00e9r\u00e9e : {speed} kilom\u00e8tres heure. Voulez-vous l'appliquer ?",
}

AUTOMATIC_ANNOUNCEMENT_TEMPLATES = {
    "pedestrian_stop": "Pi\u00e9ton d\u00e9tect\u00e9. Arr\u00eat imm\u00e9diat.",
    "stop_sign":        "Stop d\u00e9tect\u00e9. Arr\u00eat imm\u00e9diat.",
    "red_light":         "Feu rouge d\u00e9tect\u00e9. Arr\u00eat imm\u00e9diat.",
}

_last_announced_speed = {"value": None}
_pending_suggestion = {"speed": None, "question": None}


def propose_or_announce(suggested_speed, reason_code=None):
    """
    Call this after classify_full() runs on a new frame.
    Returns (spoken_text, audio_path, needs_confirmation), or (None, None, False) if nothing
    changed since last time. needs_confirmation: True -> still waiting on a driver "yes";
    False -> already applied automatically (a hard safety stop), nothing to confirm.
    """
    if suggested_speed == _last_announced_speed["value"]:
        return None, None, False

    _last_announced_speed["value"] = suggested_speed

    if reason_code in AUTOMATIC_REASON_CODES:
        text = AUTOMATIC_ANNOUNCEMENT_TEMPLATES[reason_code]
        audio_path = speak(text, output_path="speed_announcement.mp3", language_code=ANNOUNCEMENT_LANGUAGE)
        return text, audio_path, False

    template = SUGGESTION_QUESTION_TEMPLATES.get(reason_code, SUGGESTION_QUESTION_TEMPLATES[None])
    text = template.format(speed=suggested_speed)
    audio_path = speak(text, output_path="speed_announcement.mp3", language_code=ANNOUNCEMENT_LANGUAGE)

    _pending_suggestion["speed"] = suggested_speed
    _pending_suggestion["question"] = text
    return text, audio_path, True

def confirm_pending_suggestion(confirmation_text, vehicle_state):
    """
    Applies (or cancels) whatever propose_or_announce() last asked about.
    Returns (updated_vehicle_state, response_text).
    """
    if _pending_suggestion["speed"] is None:
        return vehicle_state, "There is no pending suggestion to confirm."

    confirmed, spoken_reply = _judge_yes_no(confirmation_text, _pending_suggestion["question"])
    if confirmed:
        vehicle_state["cruise_speed_kmh"] = _pending_suggestion["speed"]

    _append_history(confirmation_text, spoken_reply)

    _pending_suggestion["speed"] = None
    _pending_suggestion["question"] = None
    return vehicle_state, spoken_reply


if "vehicle_state" not in dir():
    vehicle_state = {
        "cruise_speed_kmh": 50,
        "speed_limit_kmh": None,
        "eco_mode": False,
        "safety_distance_level": 2,
        "window_open": True,
    }
if "result" in dir():
    spoken_text, audio_path, needs_confirmation = propose_or_announce(result["final_speed_kmh"], result["reason_code"])
    if spoken_text:
        print(f"Assistant says: {spoken_text}")
        play_audio_blocking(audio_path)

        if needs_confirmation:
            print("Waiting for your confirmation (say yes/no, in any language)...")
            confirm_audio_path = record_audio(duration=4, output_path="suggestion_confirmation.wav")
            confirmation_text, confirm_language = transcribe_audio(confirm_audio_path)
            print(f"Whisper heard ({confirm_language}): {confirmation_text!r}")

            vehicle_state, confirm_response = confirm_pending_suggestion(confirmation_text, vehicle_state)
            print(f"Assistant says: {confirm_response}")

            confirm_audio = speak(confirm_response, output_path="suggestion_confirm_response.mp3", language_code=confirm_language)
            play_audio_blocking(confirm_audio)
    else:
        print("Assistant stays silent (speed unchanged)")


In [ ]:
WAKE_PHRASE = "Hi there"
EXIT_PHRASE = "bye assistant"
WAKE_CHUNK_SECONDS = 4        # short clips while just waiting for the wake word
CONVERSATION_CHUNK_SECONDS = 5  # longer once you're actually mid-conversation

def _contains_phrase(text, phrase):
    return phrase.lower() in text.lower()


def run_voice_assistant_loop():
    """
    Runs forever: listens in short chunks for WAKE_PHRASE, then holds a full
    conversation (with propose->confirm on real commands) until it hears
    EXIT_PHRASE, then goes back to listening for the wake word.
    Stop the whole thing with the notebook's Interrupt/Stop button.
    """
    print(f'Listening for wake word ("{WAKE_PHRASE}")... say "{EXIT_PHRASE}" any time to end a conversation.')

    vehicle_state = {
        "cruise_speed_kmh": 50,
        "speed_limit_kmh": None,
        "eco_mode": False,
        "safety_distance_level": 2,
        "window_open": True,
    }

    listening_for_wake_word = True

    while True:
        if listening_for_wake_word:
            audio_path = record_audio(duration=WAKE_CHUNK_SECONDS, output_path="wake_check.wav")
            text, _ = transcribe_audio(audio_path)
            if _contains_phrase(text, WAKE_PHRASE):
                print("Wake word detected -- listening.")
                play_audio_blocking(speak("Yes?", output_path="wake_ack.mp3", language_code="en"))
                listening_for_wake_word = False
            continue

        # --- in conversation ---
        audio_path = record_audio(duration=CONVERSATION_CHUNK_SECONDS, output_path="conversation_turn.wav")
        text, detected_language = transcribe_audio(audio_path)

        if not text:
            continue  # caught silence -- stay in conversation mode, keep listening

        if _contains_phrase(text, EXIT_PHRASE):
            bye_text = "Goodbye." if detected_language == "en" else "Au revoir."
            play_audio_blocking(speak(bye_text, output_path="bye.mp3", language_code=detected_language))
            print("Conversation ended -- back to listening for the wake word.\n")
            listening_for_wake_word = True
            continue

        print(f"\nWhisper heard ({detected_language}): {text!r}")
        response_text, vehicle_state, needs_confirmation = understand_and_respond(text, vehicle_state)
        print(f"Assistant: {response_text}")
        play_audio_blocking(speak(response_text, language_code=detected_language))

        if needs_confirmation:
            print("Waiting for your confirmation...")
            confirm_audio_path = record_audio(duration=4, output_path="confirmation.wav")
            confirmation_text, confirm_language = transcribe_audio(confirm_audio_path)
            print(f"Whisper heard ({confirm_language}): {confirmation_text!r}")
            vehicle_state, confirm_response = confirm_pending_command(confirmation_text, vehicle_state)
            print(f"Assistant: {confirm_response}")
            play_audio_blocking(speak(confirm_response, language_code=confirm_language))

        print("\nUpdated vehicle state:", vehicle_state)


run_voice_assistant_loop()